In [1]:
import os
import sys
from pathlib import Path
print(os.getcwd())

ROOT_DIR = Path(os.getcwd()).parent.parent
print(ROOT_DIR)
sys.path.insert(0, str(ROOT_DIR))

/home/tinhanhnguyen/Desktop/HK7/Capstone/CAPSTONE_PROJECT/agentic_ai/test
/home/tinhanhnguyen/Desktop/HK7/Capstone/CAPSTONE_PROJECT


In [2]:
from agentic_ai.tools.clients import *
from agentic_ai.tools.schema.artifact import ImageObjectInterface, SegmentObjectInterface, VideoInterface
from agentic_ai.tools.type import *
from agentic_ai.agent import (
    code_act_prompt,
    WorkerCodeVideoAgent,
    AgentDecision,
    AgentInput,
    AgentOutput,
    AgentStream,
    AgentThinking,
    AgentStreamStructuredOutput,
    ToolCallResult,
    ToolCall
)
from agentic_ai.worker.executor import build_worker_executor

In [3]:
# CONSTANT
MILVUS_HOST = "localhost"
MILVUS_PORT = 19530
MILVUS_URI = f"http://{MILVUS_HOST}:{MILVUS_PORT}"

IMAGE_COLLECTION_NAME = "image_milvus"
IMAGE_VISUAL_PARAM = {
    "type_config": "dense",
    "dimension": 512,
    "metric_type": "COSINE",
    "index_type": "HNSW",
    "description": "Image embeddings for video frames",
    "index_params": {"M": 64, "efConstruction": 100},
}
IMAGE_CAPTION_DENSE_PARAM = {
    "type_config": "dense",
    "dimension": 768,
    "metric_type": "COSINE",
    "index_type": "HNSW",
    "description": "Text embeddings for image captions",
    "index_params": {"M": 64, "efConstruction": 100},
}
IMAGE_CAPTION_SPARSE_PARAM = {
    "type_config": "sparse",
    "dimension": 1000000,  # ignored at runtime
    "metric_type": "BM25",
    "index_type": "SPARSE_INVERTED_INDEX",
    "description": "BM25 sparse index for image captions",
    "index_params": {"inverted_index_algo": "DAAT_MAXSCORE"},
}
IMAGE_VISUAL_FIELD = "visual_embedding_field"
IMAGE_CAPTION_FIELD = "caption_embedding_field"
IMAGE_SPARSE_FIELD = "caption_sparse_embedding_field"


SEGMENT_CAPTION_COLLECTION_NAME = "segment_milvus"
SEGMENT_CAPTION_DENSE_PARAM = {
    "type_config": "dense",
    "dimension": 768,
    "metric_type": "COSINE",
    "index_type": "HNSW",
    "description": "Text embeddings for segment captions",
    "index_params": {"M": 64, "efConstruction": 100},
}
SEGMENT_CAPTION_SPARSE_PARAM = {
    "type_config": "sparse",
    "dimension": 1000000,  # ignored at runtime
    "metric_type": "BM25",
    "index_type": "SPARSE_INVERTED_INDEX",
    "description": "BM25 sparse index for segment captions",
    "index_params": {"inverted_index_algo": "DAAT_MAXSCORE"},
}

SEGMENT_DENSE_FIELD = "caption_embedding_field"
SEGMENT_SPARSE_FIELD = "caption_sparse_embedding_field"



In [4]:
image_milvus_client = ImageMilvusClient(
    uri=MILVUS_URI,
    collection_name=IMAGE_COLLECTION_NAME,
    visual_param=IMAGE_VISUAL_PARAM,
    caption_param=IMAGE_CAPTION_DENSE_PARAM,
    sparse_param=IMAGE_CAPTION_SPARSE_PARAM,
    visual_ann_field=IMAGE_VISUAL_FIELD,
    caption_ann_field=IMAGE_CAPTION_FIELD,
    sparse_field=IMAGE_SPARSE_FIELD,
)

segment_caption_client = SegmentCaptionImageMilvusClient(
    uri=MILVUS_URI,
    collection_name=SEGMENT_CAPTION_COLLECTION_NAME,
    dense_param=SEGMENT_CAPTION_DENSE_PARAM,
    sparse_param=SEGMENT_CAPTION_SPARSE_PARAM,
    dense_field=SEGMENT_DENSE_FIELD,
    sparse_field=SEGMENT_SPARSE_FIELD,
)

await image_milvus_client.connect()
await segment_caption_client.connect()

In [5]:
image_emebedding_settings = ImageEmbeddingSettings(
    model_name='open_clip',
    device='cuda',
    batch_size=32
)
text_emebedding_settings = TextEmbeddingSettings(
    model_name='mmbert',
    device='cuda',
    batch_size=32
)

img_txt_client = ImageEmbeddingClient(base_url='http://localhost:8003')
txt_client = TextEmbeddingClient(base_url='http://localhost:8005')


external_client = ExternalEncodeClient(img_text_client=img_txt_client, img_text_settings=image_emebedding_settings, txt_settings=text_emebedding_settings, txt_client=txt_client)

await external_client.connect()

2025-11-04 21:35:14.151 | INFO     | agentic_ai.tools.clients.external.base:load_model:126 - service-image-embedding_model_loaded
2025-11-04 21:35:14.155 | INFO     | agentic_ai.tools.clients.external.base:load_model:126 - service-text-embedding_model_loaded


In [6]:
MINIO_HOST = "localhost"       # use localhost when running locally
MINIO_PORT = 9000
MINIO_USER = "minioadmin"
MINIO_PASSWORD = "minioadmin"
MINIO_ACCESS_KEY = "minioadmin"
MINIO_SECRET_KEY = "minioadmin"
MINIO_SECURE = False  
MINIO_ENDPOINT = f"{MINIO_HOST}:{MINIO_PORT}"

storage_client = StorageClient(
    host=MINIO_HOST,
    port=MINIO_PORT,#type:ignore
    access_key=MINIO_ACCESS_KEY,
    secret_key=MINIO_SECRET_KEY,
    secure=MINIO_SECURE,
)

storage_client._ensure_bucket('anonymous')

In [7]:
from sqlalchemy import text

POSTGRE_USER = "prefect"
POSTGRE_PASSWORD = "prefect"
POSTGRE_HOST = "localhost"       # use localhost for local setup
POSTGRE_PORT = 5432
POSTGRE_DB = "prefect"

POSTGRE_DATABASE_URL = (
    f"postgresql+asyncpg://{POSTGRE_USER}:{POSTGRE_PASSWORD}"
    f"@{POSTGRE_HOST}:{POSTGRE_PORT}/{POSTGRE_DB}"
)
postgres_client = PostgresClient(
    database_url=POSTGRE_DATABASE_URL
)
async with postgres_client.get_session() as session:
    result = await session.execute(text("SELECT version();"))
    version = result.scalar_one()
    print(f"🗄️ PostgreSQL connection successful.\n   Version: {version}")

🗄️ PostgreSQL connection successful.
   Version: PostgreSQL 14.19 (Debian 14.19-1.pgdg13+1) on x86_64-pc-linux-gnu, compiled by gcc (Debian 14.2.0-19) 14.2.0, 64-bit


In [8]:
from google.genai import types
from llama_index.llms.google_genai import GoogleGenAI
from dotenv import load_dotenv
load_dotenv()


generation_config = types.GenerateContentConfig(
    temperature=0.7,
    thinking_config=types.ThinkingConfig(thinking_budget=1024, include_thoughts=True)
)

llm = GoogleGenAI(
    model='gemini-2.5-flash',
    generation_config=generation_config   
)

tool_factory = ToolFactory(
    image_milvus_client=image_milvus_client,
    segment_milvus_client=segment_caption_client,
    postgres_client=postgres_client,
    minio_client=storage_client,
    external_client=external_client,
    llm_as_tools=llm
)

In [9]:
tools = tool_factory.get_all_tools()

get_video_from_segment
get_video_from_image
get_asr_from_video
get_all_segment_info_from_video_interface
get_segments
get_images
extract_frames_by_time_window
extract_frame_time
frame_to_timecode
timecode_to_frame
from_index_to_time
from_range_index_to_range_time
from_time_to_index
from_range_time_to_range_index
read_image
read_segment
get_related_asr_from_image
get_related_asr_from_segment
get_images_from_visual_query
get_images_from_caption_query
get_segments_from_event_query
get_images_from_multimodal_query
find_similar_images_from_image
enhance_visual_query
enhance_textual_query
caption_new_image


In [10]:
USER = "anonymous"
LIST_VIDEO_IDS = ["video1_111", "video2_222"]

get_all_functools = tool_factory.get_all_tools_functool(
    user_id=USER,
    list_video_id=LIST_VIDEO_IDS,
    agent_bucket='agent_test',
    agent_object_folder='agent_worker_small'
)

get_all_tools_normal = tool_factory.get_all_tools_normal(
    user_id=USER,
    list_video_id=LIST_VIDEO_IDS,
    agent_bucket='agent_test',
    agent_object_folder='agent_worker_small'
)

In [11]:
test_tools = [
    'get_images_from_visual_query',
    'get_segments_from_event_query',
    'read_image',
    'read_segment',
    'enhance_visual_query',
    'enhance_textual_query',
    'get_segments',
    'get_images',
    'frame_to_timecode',
    'timecode_to_frame',
    'from_index_to_time',
    'get_related_asr_from_image',
    'get_related_asr_from_segment',
    'extract_frame_time',
]

get_test_tools_func = list(
    filter(lambda x: x[0] in test_tools, get_all_functools.items())
)
get_test_tools_normal = dict(list(
    filter(lambda x: x[0] in test_tools, get_all_tools_normal.items())
))

globals_dependency = tool_factory.dependency_map
globals_dependency.update(
    {
        'ImageObjectInterface': ImageObjectInterface,
        'SegmentObjectInterface': SegmentObjectInterface,
        'VideoInterface': VideoInterface
    }
)

env_fn = build_worker_executor(
    tools_bindings=get_test_tools_normal,
    extra_globals=globals_dependency
)

In [19]:
code_exec = """
print("Hi")

async def print_ski():
    print("Hello")

await print_ski()
"""

In [20]:
result = await env_fn(code_exec)

Hi
Hello
Hi


In [21]:
result

'stdout:\nHi\nHello\nHi\n\nexception:\nSyntaxError: \'await\' outside function (<sandbox>, line 6)\nTraceback (most recent call last):\n  File "/home/tinhanhnguyen/Desktop/HK7/Capstone/CAPSTONE_PROJECT/agentic_ai/worker/executor.py", line 185, in execute\n    )\n      \n  File "<sandbox>", line 6\nSyntaxError: \'await\' outside function'